# CardioVUS-KCNH2 — EDA funcional del mapa de variantes

**Propósito general:** comprender qué mide el experimento de MaveDB, verificar la calidad bioinformática de las variantes y caracterizar patrones funcionales de KCNH2 antes de construir modelos predictivos.

El notebook analiza tres niveles:

1. **Experimental:** score, controles e incertidumbre.
2. **Bioinformático:** cobertura, normalización y correspondencia con `NP_000229.1`.
3. **Biológico:** sensibilidad por residuo y efecto de propiedades fisicoquímicas.

> El score funcional no equivale automáticamente a patogenicidad clínica. El ensayo evalúa principalmente expresión o tráfico superficial y no necesariamente gating, conductancia o riesgo clínico.

## 1. Configuración reproducible — propósito: asegurar rutas, versiones y semillas consistentes

Esta sección localiza la raíz del repositorio, define las rutas, crea las carpetas de resultados, fija una semilla y confirma que todos los archivos requeridos estén disponibles.

In [ ]:
from __future__ import annotations

import json
import platform
import re
import sys
import warnings
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from Bio import SeqIO
from Bio.Align import substitution_matrices
from IPython.display import Markdown, display
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "src").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. "
        "Ejecuta el notebook dentro de byte2beat-cardiovus."
    )


ROOT_DIR = find_project_root(Path.cwd())

RAW_MAVEDB_DIR = ROOT_DIR / "data" / "raw" / "mavedb"
RAW_REFERENCE_DIR = ROOT_DIR / "data" / "raw" / "reference"
INTERIM_DIR = ROOT_DIR / "data" / "interim"
EDA_REPORTS_DIR = ROOT_DIR / "reports" / "eda"
FIGURES_DIR = ROOT_DIR / "outputs" / "figures" / "eda"

SCORES_PATH = RAW_MAVEDB_DIR / "scores.csv"
METADATA_PATH = RAW_MAVEDB_DIR / "score_set_metadata.json"
MAPPED_VARIANTS_PATH = RAW_MAVEDB_DIR / "mapped_variants.json"
FASTA_PATH = RAW_REFERENCE_DIR / "NP_000229.1.fasta"
MANIFEST_PATH = ROOT_DIR / "data" / "data_manifest.csv"

QC_PARQUET_PATH = INTERIM_DIR / "kcnh2_variants_qc.parquet"
NORMALIZED_PARQUET_PATH = (
    INTERIM_DIR / "kcnh2_variants_normalized.parquet"
)

EDA_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    SCORES_PATH,
    METADATA_PATH,
    MAPPED_VARIANTS_PATH,
    FASTA_PATH,
    MANIFEST_PATH,
    QC_PARQUET_PATH,
    NORMALIZED_PARQUET_PATH,
]

missing_files = [path for path in REQUIRED_FILES if not path.exists()]

if missing_files:
    formatted = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(f"Faltan archivos requeridos:\n{formatted}")

print(f"Repository root: {ROOT_DIR}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"SciPy: {scipy.__version__}")

## 2. Contexto biológico — propósito: conectar KCNH2 con la repolarización cardíaca y delimitar el alcance del ensayo

`KCNH2` codifica el canal Kv11.1/hERG, que contribuye a la corriente de potasio `IKr` durante la repolarización cardíaca.

```text
Variante en KCNH2
        ↓
Cambio en Kv11.1
        ↓
Plegamiento, estabilidad, tráfico, gating o conductancia
        ↓
Alteración de IKr
        ↓
Cambio en la repolarización y el intervalo QT
```

El dataset mide principalmente cuánto canal alcanza la superficie celular. Una variante puede traficar correctamente y aun así presentar alteraciones electrofisiológicas.

## 3. Procedencia y referencia — propósito: demostrar qué archivos y secuencia sostienen el análisis

Se revisa el manifiesto con sus hashes SHA-256 y se valida que la referencia corresponda a `NP_000229.1` con 1.159 aminoácidos.

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)

display(
    manifest_df[
        [
            "source",
            "source_identifier",
            "file_role",
            "relative_path",
            "size_bytes",
            "sha256",
        ]
    ]
)

reference_record = SeqIO.read(FASTA_PATH, "fasta")
reference_sequence = str(reference_record.seq).upper()

reference_summary = pd.DataFrame(
    {
        "attribute": ["ID", "length", "description"],
        "value": [
            reference_record.id,
            len(reference_sequence),
            reference_record.description,
        ],
    }
)

display(reference_summary)

assert reference_record.id == "NP_000229.1"
assert len(reference_sequence) == 1159

print("La referencia cumple ID y longitud esperados.")

## 4. Diseño experimental y metadatos — propósito: determinar qué representa el score antes de interpretarlo

Se inspeccionan los metadatos para localizar evidencia sobre:

- definición y dirección del score;
- normalización y controles;
- replicados;
- `se`, `LLR` e intervalos;
- target experimental;
- publicación y licencia.

No se debe llamar “dañina” a una variante únicamente porque tenga un score bajo sin confirmar primero la escala.

In [ ]:
with METADATA_PATH.open("r", encoding="utf-8") as file:
    score_set_metadata = json.load(file)


def flatten_json(
    obj: Any,
    parent_key: str = "root",
    output: dict[str, Any] | None = None,
) -> dict[str, Any]:
    if output is None:
        output = {}

    if isinstance(obj, dict):
        for key, value in obj.items():
            flatten_json(value, f"{parent_key}.{key}", output)

    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            flatten_json(value, f"{parent_key}[{index}]", output)

    else:
        output[parent_key] = obj

    return output


flat_metadata = flatten_json(score_set_metadata)

if isinstance(score_set_metadata, dict):
    top_level_summary = pd.DataFrame(
        {
            "top_level_key": list(score_set_metadata.keys()),
            "value_type": [
                type(score_set_metadata[key]).__name__
                for key in score_set_metadata
            ],
        }
    )
    display(top_level_summary)

keywords = [
    "score",
    "normal",
    "control",
    "replicate",
    "method",
    "target",
    "sequence",
    "publication",
    "doi",
    "pmid",
    "license",
    "assay",
    "expression",
    "traffic",
]

metadata_hits = []

for path, value in flat_metadata.items():
    searchable = f"{path} {value}".lower()

    if any(keyword in searchable for keyword in keywords):
        text = str(value)
        metadata_hits.append(
            {
                "path": path,
                "value": text[:500] + ("..." if len(text) > 500 else ""),
            }
        )

metadata_hits_df = pd.DataFrame(metadata_hits)

print(f"Campos planos totales: {len(flat_metadata):,}")
print(
    "Campos potencialmente informativos: "
    f"{len(metadata_hits_df):,}"
)

display(metadata_hits_df.head(100))

metadata_hits_df.to_csv(
    EDA_REPORTS_DIR / "metadata_keyword_hits.csv",
    index=False,
)

In [ ]:
metadata_text = json.dumps(score_set_metadata, ensure_ascii=False)

doi_candidates = sorted(
    set(
        re.findall(
            r"10\.\d{4,9}/[-._;()/:A-Z0-9]+",
            metadata_text,
            flags=re.IGNORECASE,
        )
    )
)

accession_candidates = sorted(
    set(
        re.findall(
            r"\b(?:NP|NM|ENSP|ENST)_?\d+(?:\.\d+)?\b",
            metadata_text,
        )
    )
)

print("DOI candidates:", doi_candidates[:20])
print("Reference accession candidates:", accession_candidates[:20])
print("Metadata contains KCNH2:", "KCNH2" in metadata_text)
print(
    "Metadata contains NP_000229.1:",
    "NP_000229.1" in metadata_text,
)

### 4.1 Lista de verificación — propósito: separar evidencia confirmada de supuestos pendientes

Esta tabla debe completarse después de revisar los metadatos y los métodos del artículo. El código no inventa respuestas cuando la información no está explícita.

In [ ]:
checklist_path = (
    EDA_REPORTS_DIR / "experimental_design_checklist.csv"
)

if checklist_path.exists():
    experimental_questions = pd.read_csv(checklist_path)
else:
    experimental_questions = pd.DataFrame(
        {
            "question": [
                "¿Qué mide exactamente score?",
                "¿Valores altos significan mayor expresión superficial?",
                "¿Cómo se normalizó el score?",
                "¿Cuál fue el comportamiento esperado del WT?",
                "¿Qué controles loss-of-function se utilizaron?",
                "¿Cuántos replicados hubo?",
                "¿Qué representan se y LLR?",
                "¿Qué regiones de KCNH2 fueron cubiertas?",
                "¿El target coincide con NP_000229.1?",
                "¿Por qué el CSV tiene más filas que el número descrito en el artículo?",
            ],
            "status": ["PENDIENTE_DE_CONFIRMAR"] * 10,
            "conclusion_confirmada": [""] * 10,
            "source_or_evidence": [""] * 10,
        }
    )
    experimental_questions.to_csv(
        checklist_path,
        index=False,
    )

display(experimental_questions)
print(f"Checklist: {checklist_path}")

## 5. Carga de datos — propósito: mantener separadas la fuente original, la auditoría y la tabla de modelamiento

- `raw_scores`: CSV original.
- `qc_df`: todas las variantes con controles de calidad.
- `missense_df`: variantes missense simples aptas para el modelamiento.

In [ ]:
raw_scores = pd.read_csv(SCORES_PATH, low_memory=False)
qc_df = pd.read_parquet(QC_PARQUET_PATH)
missense_df = pd.read_parquet(NORMALIZED_PARQUET_PATH)

missense_df["position"] = (
    missense_df["position"].astype("int16")
)

dataset_summary = pd.DataFrame(
    {
        "dataset": ["raw_scores", "qc_df", "missense_df"],
        "rows": [
            len(raw_scores),
            len(qc_df),
            len(missense_df),
        ],
        "columns": [
            raw_scores.shape[1],
            qc_df.shape[1],
            missense_df.shape[1],
        ],
        "purpose": [
            "Fuente original",
            "Auditoría de todas las variantes",
            "Modelamiento missense",
        ],
    }
)

display(dataset_summary)

display(
    pd.DataFrame(
        {
            "column": raw_scores.columns,
            "dtype": raw_scores.dtypes.astype(str),
        }
    )
)

display(
    missense_df[
        [
            "variant_id",
            "hgvs_pro_normalized",
            "position",
            "score_numeric",
            "se",
            "LLR",
        ]
    ].head(10)
)

## 6. Control de calidad bioinformático — propósito: confirmar que cada score corresponde a una variante válida

Se verifican scores, clases de variantes, concordancia con el FASTA, duplicados y missing values. Un mismatch podría indicar una isoforma diferente o una numeración incompatible.

In [ ]:
variant_class_summary = (
    qc_df["variant_class"]
    .value_counts(dropna=False)
    .rename_axis("variant_class")
    .reset_index(name="count")
)

variant_class_summary["percentage"] = (
    variant_class_summary["count"] / len(qc_df) * 100
)

display(variant_class_summary)

qc_checks = pd.DataFrame(
    {
        "check": [
            "Scores numéricos presentes",
            "Variantes missense simples",
            "Variantes múltiples",
            "Missense con match al FASTA",
            "Missense con mismatch al FASTA",
            "IDs proteicos duplicados",
        ],
        "count": [
            int(qc_df["score_numeric"].notna().sum()),
            int(
                qc_df["variant_class"]
                .eq("simple_missense")
                .sum()
            ),
            int(
                qc_df["variant_class"]
                .eq("multi_variant")
                .sum()
            ),
            int(qc_df["ref_match"].eq(True).sum()),
            int(qc_df["ref_match"].eq(False).sum()),
            int(
                qc_df["protein_variant_duplicate"].sum()
            ),
        ],
    }
)

display(qc_checks)

missing_summary = (
    qc_df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(qc_df) * 100
)

display(
    missing_summary.loc[
        missing_summary["missing_count"] > 0
    ].sort_values(
        "missing_percentage",
        ascending=False,
    )
)

assert qc_df["score_numeric"].notna().all()
assert missense_df["ref_match"].eq(True).all()
assert not missense_df[
    "protein_variant_duplicate"
].any()

variant_class_summary.to_csv(
    EDA_REPORTS_DIR / "variant_class_summary.csv",
    index=False,
)

## 7. Controles y dirección de la escala — propósito: inferir empíricamente cómo se comporta el score

Se comparan variantes:

- `synonymous`: aproximación a comportamiento WT-like;
- `stop_gained`: enriquecidas en truncamiento;
- `simple_missense`: efectos heterogéneos.

La separación entre clases ayuda a comprender la escala, pero no reemplaza la lectura de los métodos.

In [ ]:
analysis_classes = [
    "synonymous",
    "simple_missense",
    "stop_gained",
]

score_by_class = qc_df.loc[
    qc_df["variant_class"].isin(analysis_classes),
    ["variant_class", "score_numeric"],
].dropna()

class_score_summary = (
    score_by_class.groupby(
        "variant_class"
    )["score_numeric"]
    .agg(
        n="count",
        mean="mean",
        std="std",
        median="median",
        minimum="min",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
        maximum="max",
    )
    .reindex(analysis_classes)
)

display(class_score_summary)

class_score_summary.to_csv(
    EDA_REPORTS_DIR
    / "score_summary_by_variant_class.csv"
)

synonymous_median = class_score_summary.loc[
    "synonymous",
    "median",
]

stop_median = class_score_summary.loc[
    "stop_gained",
    "median",
]

if synonymous_median > stop_median:
    message = (
        "Los controles synonymous presentan una mediana mayor "
        "que stop-gained. Esto apoya que valores altos podrían "
        "representar mayor expresión superficial o un comportamiento "
        "más WT-like, pendiente de confirmación metodológica."
    )
else:
    message = (
        "Los controles no muestran la dirección esperada. "
        "La escala debe resolverse desde los métodos."
    )

display(Markdown(f"**Dirección empírica:** {message}"))

In [ ]:
for variant_class in analysis_classes:
    values = score_by_class.loc[
        score_by_class["variant_class"].eq(
            variant_class
        ),
        "score_numeric",
    ]

    plt.figure(figsize=(9, 5))
    plt.hist(values, bins=60)
    plt.xlabel("Score experimental")
    plt.ylabel("Número de variantes")
    plt.title(
        f"Distribución del score: {variant_class}\n"
        "Propósito: observar escala, asimetría y subpoblaciones"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
ordered_values = [
    score_by_class.loc[
        score_by_class["variant_class"].eq(label),
        "score_numeric",
    ].values
    for label in analysis_classes
]

plt.figure(figsize=(9, 6))
plt.boxplot(
    ordered_values,
    tick_labels=analysis_classes,
    showfliers=False,
)
plt.ylabel("Score experimental")
plt.title(
    "Comparación del score entre clases\n"
    "Propósito: evaluar separación entre controles y missense"
)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 7.1 Comparación estadística — propósito: cuantificar diferencias y solapamiento entre clases

Se utiliza Kruskal–Wallis y comparaciones Mann–Whitney porque no se asume normalidad. El tamaño de efecto por rangos muestra cuánto tiende una clase a presentar valores mayores que otra.

In [ ]:
kruskal_result = stats.kruskal(*ordered_values)

print(
    f"Kruskal-Wallis H={kruskal_result.statistic:,.4f}, "
    f"p={kruskal_result.pvalue:.3e}"
)


def holm_adjust(
    p_values: Iterable[float],
) -> np.ndarray:
    p_values = np.asarray(
        list(p_values),
        dtype=float,
    )

    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running_max = 0.0

    for rank, index in enumerate(order):
        adjusted_value = (
            len(p_values) - rank
        ) * p_values[index]

        running_max = max(
            running_max,
            adjusted_value,
        )

        adjusted[index] = min(
            running_max,
            1.0,
        )

    return adjusted


pairwise_results = []

for index, class_a in enumerate(
    analysis_classes
):
    for class_b in analysis_classes[
        index + 1 :
    ]:
        values_a = score_by_class.loc[
            score_by_class[
                "variant_class"
            ].eq(class_a),
            "score_numeric",
        ].values

        values_b = score_by_class.loc[
            score_by_class[
                "variant_class"
            ].eq(class_b),
            "score_numeric",
        ].values

        test = stats.mannwhitneyu(
            values_a,
            values_b,
            alternative="two-sided",
        )

        rank_biserial = (
            2
            * test.statistic
            / (len(values_a) * len(values_b))
        ) - 1

        pairwise_results.append(
            {
                "class_a": class_a,
                "class_b": class_b,
                "n_a": len(values_a),
                "n_b": len(values_b),
                "median_a": np.median(values_a),
                "median_b": np.median(values_b),
                "u_statistic": test.statistic,
                "p_value": test.pvalue,
                "rank_biserial_effect": (
                    rank_biserial
                ),
            }
        )

pairwise_df = pd.DataFrame(
    pairwise_results
)

pairwise_df["p_holm"] = holm_adjust(
    pairwise_df["p_value"]
)

display(pairwise_df)

pairwise_df.to_csv(
    EDA_REPORTS_DIR
    / "pairwise_score_class_tests.csv",
    index=False,
)

## 8. Cobertura mutacional — propósito: distinguir tolerancia biológica de ausencia de medición

Cada posición admite hasta 19 sustituciones missense. Se calcula cuántas posiciones y sustituciones fueron evaluadas y dónde existen vacíos.

In [ ]:
all_positions = pd.DataFrame(
    {
        "position": np.arange(
            1,
            len(reference_sequence) + 1,
        )
    }
)

coverage_observed = (
    missense_df.groupby("position")
    .agg(
        n_missense=("variant_id", "count"),
        n_unique_mutants=(
            "mut_aa1",
            "nunique",
        ),
        wt_residue=("wt_aa1", "first"),
    )
    .reset_index()
)

coverage_df = all_positions.merge(
    coverage_observed,
    on="position",
    how="left",
)

coverage_df["n_missense"] = (
    coverage_df["n_missense"]
    .fillna(0)
    .astype("int16")
)

coverage_df["n_unique_mutants"] = (
    coverage_df["n_unique_mutants"]
    .fillna(0)
    .astype("int16")
)

coverage_df["wt_residue"] = [
    reference_sequence[position - 1]
    for position in coverage_df["position"]
]

coverage_df["coverage_fraction"] = (
    coverage_df["n_missense"] / 19
)

coverage_df["fully_covered"] = (
    coverage_df["n_missense"].eq(19)
)

coverage_summary = pd.Series(
    {
        "protein_length": len(
            reference_sequence
        ),
        "positions_with_missense": int(
            coverage_df[
                "n_missense"
            ].gt(0).sum()
        ),
        "positions_without_missense": int(
            coverage_df[
                "n_missense"
            ].eq(0).sum()
        ),
        "fully_covered_positions": int(
            coverage_df[
                "fully_covered"
            ].sum()
        ),
        "observed_missense_variants": int(
            coverage_df[
                "n_missense"
            ].sum()
        ),
        "theoretical_missense_space": (
            len(reference_sequence) * 19
        ),
        "global_coverage_fraction": (
            coverage_df[
                "n_missense"
            ].sum()
            / (
                len(reference_sequence)
                * 19
            )
        ),
    },
    name="value",
)

display(coverage_summary.to_frame())

display(
    coverage_df.sort_values(
        ["n_missense", "position"]
    ).head(30)
)

coverage_df.to_csv(
    EDA_REPORTS_DIR
    / "coverage_by_position.csv",
    index=False,
)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(
    coverage_df["position"],
    coverage_df["n_missense"],
    linewidth=0.8,
)
plt.axhline(
    19,
    linestyle="--",
    linewidth=1,
)
plt.xlabel("Posición en NP_000229.1")
plt.ylabel(
    "Número de sustituciones missense"
)
plt.title(
    "Cobertura mutacional a lo largo de KCNH2\n"
    "Propósito: identificar regiones completas y vacíos"
)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    coverage_df["n_missense"],
    bins=np.arange(-0.5, 20.5, 1),
)
plt.xlabel(
    "Sustituciones observadas por posición"
)
plt.ylabel("Número de posiciones")
plt.title(
    "Distribución de cobertura por residuo\n"
    "Propósito: medir cercanía a la saturación de 19 cambios"
)
plt.tight_layout()
plt.show()

## 9. Sensibilidad por residuo — propósito: localizar posiciones con scores consistentemente bajos o dependientes del aminoácido mutante

La mediana resume tolerancia global de una posición. El IQR resume cuánto cambia el resultado según el aminoácido introducido.

In [ ]:
position_summary = (
    missense_df.groupby("position")
    .agg(
        wt_residue=("wt_aa1", "first"),
        n_variants=("variant_id", "count"),
        mean_score=(
            "score_numeric",
            "mean",
        ),
        median_score=(
            "score_numeric",
            "median",
        ),
        std_score=(
            "score_numeric",
            "std",
        ),
        min_score=("score_numeric", "min"),
        q1_score=(
            "score_numeric",
            lambda x: x.quantile(0.25),
        ),
        q3_score=(
            "score_numeric",
            lambda x: x.quantile(0.75),
        ),
        max_score=("score_numeric", "max"),
        median_se=("se", "median"),
    )
    .reset_index()
)

position_summary["iqr_score"] = (
    position_summary["q3_score"]
    - position_summary["q1_score"]
)

print("Posiciones con menor mediana:")
display(
    position_summary.sort_values(
        "median_score"
    ).head(20)
)

print("Posiciones con mayor mediana:")
display(
    position_summary.sort_values(
        "median_score",
        ascending=False,
    ).head(20)
)

position_summary.to_csv(
    EDA_REPORTS_DIR
    / "position_score_summary.csv",
    index=False,
)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(
    position_summary["position"],
    position_summary["median_score"],
    linewidth=0.8,
)
plt.xlabel("Posición en NP_000229.1")
plt.ylabel("Mediana del score")
plt.title(
    "Score mediano por residuo\n"
    "Propósito: localizar posiciones con baja o alta tolerancia"
)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(
    position_summary["position"],
    position_summary["iqr_score"],
    linewidth=0.8,
)
plt.xlabel("Posición en NP_000229.1")
plt.ylabel("IQR del score")
plt.title(
    "Variabilidad entre sustituciones de una posición\n"
    "Propósito: detectar efectos dependientes del mutante"
)
plt.tight_layout()
plt.show()

In [ ]:
canonical_amino_acids = list(
    "ACDEFGHIKLMNPQRSTVWY"
)

mutational_matrix = (
    missense_df.pivot_table(
        index="mut_aa1",
        columns="position",
        values="score_numeric",
        aggfunc="mean",
    )
    .reindex(canonical_amino_acids)
)

plt.figure(figsize=(18, 6))
image = plt.imshow(
    mutational_matrix,
    aspect="auto",
    interpolation="nearest",
)
plt.colorbar(
    image,
    label="Score experimental",
)
plt.yticks(
    ticks=np.arange(
        len(canonical_amino_acids)
    ),
    labels=canonical_amino_acids,
)
plt.xlabel("Posición en NP_000229.1")
plt.ylabel("Aminoácido mutante")
plt.title(
    "Mapa global de efectos missense\n"
    "Propósito: visualizar regiones sensibles y patrones"
)
plt.tight_layout()
plt.show()

## 10. Propiedades fisicoquímicas — propósito: evaluar si cambios moleculares interpretables explican parte del score

Se analizan hidrofobicidad, carga, polaridad y BLOSUM62. Estas variables serán el baseline interpretable que posteriormente se comparará con ESM-2.

In [ ]:
HYDROPHOBICITY_KD = {
    "A": 1.8,
    "R": -4.5,
    "N": -3.5,
    "D": -3.5,
    "C": 2.5,
    "Q": -3.5,
    "E": -3.5,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "L": 3.8,
    "K": -3.9,
    "M": 1.9,
    "F": 2.8,
    "P": -1.6,
    "S": -0.8,
    "T": -0.7,
    "W": -0.9,
    "Y": -1.3,
    "V": 4.2,
}

CHARGE_CLASS = {
    "D": "negative",
    "E": "negative",
    "K": "positive",
    "R": "positive",
    "H": "positive",
    "A": "neutral",
    "C": "neutral",
    "F": "neutral",
    "G": "neutral",
    "I": "neutral",
    "L": "neutral",
    "M": "neutral",
    "N": "neutral",
    "P": "neutral",
    "Q": "neutral",
    "S": "neutral",
    "T": "neutral",
    "V": "neutral",
    "W": "neutral",
    "Y": "neutral",
}

POLARITY_CLASS = {
    aa: (
        "polar"
        if aa in set("RNDQEKHSTY")
        else "nonpolar"
    )
    for aa in HYDROPHOBICITY_KD
}

blosum62 = substitution_matrices.load(
    "BLOSUM62"
)


def get_blosum_score(
    wt: str,
    mut: str,
) -> float:
    try:
        return float(blosum62[wt, mut])
    except (IndexError, KeyError):
        return float(blosum62[mut, wt])


feature_df = missense_df.copy()

feature_df["wt_hydrophobicity"] = (
    feature_df["wt_aa1"].map(
        HYDROPHOBICITY_KD
    )
)

feature_df["mut_hydrophobicity"] = (
    feature_df["mut_aa1"].map(
        HYDROPHOBICITY_KD
    )
)

feature_df["delta_hydrophobicity"] = (
    feature_df["mut_hydrophobicity"]
    - feature_df["wt_hydrophobicity"]
)

feature_df[
    "abs_delta_hydrophobicity"
] = feature_df[
    "delta_hydrophobicity"
].abs()

feature_df["wt_charge"] = (
    feature_df["wt_aa1"].map(
        CHARGE_CLASS
    )
)

feature_df["mut_charge"] = (
    feature_df["mut_aa1"].map(
        CHARGE_CLASS
    )
)

feature_df["charge_change"] = np.where(
    feature_df["wt_charge"].eq(
        feature_df["mut_charge"]
    ),
    "same_charge_class",
    feature_df["wt_charge"]
    + "_to_"
    + feature_df["mut_charge"],
)

feature_df["wt_polarity"] = (
    feature_df["wt_aa1"].map(
        POLARITY_CLASS
    )
)

feature_df["mut_polarity"] = (
    feature_df["mut_aa1"].map(
        POLARITY_CLASS
    )
)

feature_df["polarity_change"] = np.where(
    feature_df["wt_polarity"].eq(
        feature_df["mut_polarity"]
    ),
    "same_polarity_class",
    feature_df["wt_polarity"]
    + "_to_"
    + feature_df["mut_polarity"],
)

feature_df["blosum62"] = [
    get_blosum_score(wt, mut)
    for wt, mut in zip(
        feature_df["wt_aa1"],
        feature_df["mut_aa1"],
    )
]

display(
    feature_df[
        [
            "variant_id",
            "score_numeric",
            "delta_hydrophobicity",
            "charge_change",
            "polarity_change",
            "blosum62",
        ]
    ].head()
)

In [ ]:
correlation_rows = []

for feature in [
    "delta_hydrophobicity",
    "abs_delta_hydrophobicity",
    "blosum62",
]:
    valid = feature_df[
        [feature, "score_numeric"]
    ].dropna()

    result = stats.spearmanr(
        valid[feature],
        valid["score_numeric"],
    )

    correlation_rows.append(
        {
            "feature": feature,
            "n": len(valid),
            "spearman_rho": (
                result.statistic
            ),
            "p_value": result.pvalue,
        }
    )

physicochemical_correlations_df = (
    pd.DataFrame(correlation_rows)
)

display(
    physicochemical_correlations_df
)

physicochemical_correlations_df.to_csv(
    EDA_REPORTS_DIR
    / "physicochemical_correlations.csv",
    index=False,
)

In [ ]:
plt.figure(figsize=(9, 6))
plt.hexbin(
    feature_df[
        "abs_delta_hydrophobicity"
    ],
    feature_df["score_numeric"],
    gridsize=45,
    mincnt=1,
)
plt.colorbar(
    label="Número de variantes"
)
plt.xlabel(
    "Cambio absoluto de hidrofobicidad"
)
plt.ylabel("Score experimental")
plt.title(
    "Cambio de hidrofobicidad frente al score\n"
    "Propósito: evaluar si cambios químicos grandes alteran el tráfico"
)
plt.tight_layout()
plt.show()

In [ ]:
charge_order = (
    feature_df.groupby(
        "charge_change"
    )["score_numeric"]
    .median()
    .sort_values()
    .index
    .tolist()
)

charge_values = [
    feature_df.loc[
        feature_df["charge_change"].eq(
            label
        ),
        "score_numeric",
    ].values
    for label in charge_order
]

plt.figure(figsize=(13, 6))
plt.boxplot(
    charge_values,
    tick_labels=charge_order,
    showfliers=False,
)
plt.ylabel("Score experimental")
plt.title(
    "Score según cambio de carga\n"
    "Propósito: evaluar si introducir o eliminar carga afecta el tráfico"
)
plt.xticks(
    rotation=60,
    ha="right",
)
plt.tight_layout()
plt.show()

## 11. Incertidumbre experimental — propósito: distinguir scores precisos de estimaciones inestables

Se revisan `se`, intervalos del LLR y fuerza de evidencia. Aún no se filtran variantes; primero se caracteriza la incertidumbre.

In [ ]:
uncertainty_columns = [
    "se",
    "LLR",
    "LLR_ci_lower",
    "LLR_ci_upper",
    "LLR_evidence_strength",
]

uncertainty_missing = pd.DataFrame(
    {
        "column": uncertainty_columns,
        "missing_count": [
            qc_df[column].isna().sum()
            for column in uncertainty_columns
        ],
        "missing_percentage": [
            qc_df[column].isna().mean()
            * 100
            for column in uncertainty_columns
        ],
    }
)

display(uncertainty_missing)

qc_df["LLR_ci_width"] = (
    qc_df["LLR_ci_upper"]
    - qc_df["LLR_ci_lower"]
)

display(
    qc_df[
        ["se", "LLR", "LLR_ci_width"]
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.5,
            0.75,
            0.95,
            0.99,
        ]
    )
)

In [ ]:
se_values = qc_df["se"].dropna()

plt.figure(figsize=(9, 5))
plt.hist(se_values, bins=60)
plt.xlabel("Error estándar")
plt.ylabel("Número de variantes")
plt.title(
    "Distribución del error estándar\n"
    "Propósito: identificar estimaciones de baja precisión"
)
plt.tight_layout()
plt.show()

In [ ]:
valid_se = qc_df[
    ["score_numeric", "se"]
].dropna()

se_score_corr = stats.spearmanr(
    valid_se["score_numeric"],
    valid_se["se"],
)

print(
    "Spearman score vs se: "
    f"rho={se_score_corr.statistic:.4f}, "
    f"p={se_score_corr.pvalue:.3e}"
)

plt.figure(figsize=(9, 6))
plt.hexbin(
    valid_se["score_numeric"],
    valid_se["se"],
    gridsize=45,
    mincnt=1,
)
plt.colorbar(
    label="Número de variantes"
)
plt.xlabel("Score experimental")
plt.ylabel("Error estándar")
plt.title(
    "Score frente a incertidumbre\n"
    "Propósito: comprobar si valores extremos son menos precisos"
)
plt.tight_layout()
plt.show()

In [ ]:
evidence_counts = (
    qc_df["LLR_evidence_strength"]
    .fillna("MISSING")
    .value_counts(dropna=False)
    .rename_axis(
        "LLR_evidence_strength"
    )
    .reset_index(name="count")
)

evidence_counts["percentage"] = (
    evidence_counts["count"]
    / len(qc_df)
    * 100
)

display(evidence_counts)

missing_se_by_class = (
    qc_df.assign(
        se_missing=qc_df["se"].isna()
    )
    .groupby(
        "variant_class"
    )["se_missing"]
    .agg(["sum", "mean", "count"])
    .rename(
        columns={
            "sum": "missing_se_count",
            "mean": (
                "missing_se_fraction"
            ),
            "count": "class_count",
        }
    )
)

display(missing_se_by_class)

## 12. Score y LLR — propósito: detectar dependencia y prevenir target leakage

Si `LLR` se deriva del mismo resultado experimental, no debe utilizarse como feature para predecir `score`. Puede utilizarse para control de calidad o análisis de sensibilidad.

In [ ]:
valid_llr = qc_df[
    ["score_numeric", "LLR"]
].dropna()

spearman_llr = stats.spearmanr(
    valid_llr["score_numeric"],
    valid_llr["LLR"],
)

pearson_llr = stats.pearsonr(
    valid_llr["score_numeric"],
    valid_llr["LLR"],
)

llr_relationship = pd.DataFrame(
    {
        "metric": [
            "Spearman",
            "Pearson",
        ],
        "coefficient": [
            spearman_llr.statistic,
            pearson_llr.statistic,
        ],
        "p_value": [
            spearman_llr.pvalue,
            pearson_llr.pvalue,
        ],
        "n": [
            len(valid_llr),
            len(valid_llr),
        ],
    }
)

display(llr_relationship)

plt.figure(figsize=(9, 6))
plt.hexbin(
    valid_llr["score_numeric"],
    valid_llr["LLR"],
    gridsize=50,
    mincnt=1,
)
plt.colorbar(
    label="Número de variantes"
)
plt.xlabel("Score experimental")
plt.ylabel("LLR")
plt.title(
    "Relación entre score y LLR\n"
    "Propósito: detectar dependencia y evitar leakage"
)
plt.tight_layout()
plt.show()

## 13. `mapped_variants.json` — propósito: evaluar la futura interoperabilidad con ClinVar y GA4GH

Se inspecciona la estructura, identificadores GA4GH y accessions. La integración detallada se pospone hasta la armonización clínica.

In [ ]:
with MAPPED_VARIANTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    mapped_variants = json.load(file)


def summarize_json_node(
    obj: Any,
) -> dict[str, Any]:
    if isinstance(obj, dict):
        return {
            "root_type": "dict",
            "root_length": len(obj),
            "first_keys": list(
                obj.keys()
            )[:30],
        }

    if isinstance(obj, list):
        return {
            "root_type": "list",
            "root_length": len(obj),
            "first_item_type": (
                type(obj[0]).__name__
                if obj
                else None
            ),
            "first_item_keys": (
                list(obj[0].keys())[:30]
                if obj
                and isinstance(
                    obj[0],
                    dict,
                )
                else None
            ),
        }

    return {
        "root_type": (
            type(obj).__name__
        ),
        "root_length": None,
    }


mapped_structure = summarize_json_node(
    mapped_variants
)

display(
    pd.Series(
        mapped_structure,
        name="value",
    ).to_frame()
)

mapped_text = json.dumps(
    mapped_variants,
    ensure_ascii=False,
)

ga4gh_ids = sorted(
    set(
        re.findall(
            r"ga4gh:[A-Za-z0-9._-]+",
            mapped_text,
        )
    )
)

reference_accessions = sorted(
    set(
        re.findall(
            r"\b(?:NP|NM|NC|NG|ENSP|ENST)_?\d+(?:\.\d+)?\b",
            mapped_text,
        )
    )
)

print(
    "Unique GA4GH-like IDs:",
    f"{len(ga4gh_ids):,}",
)
print(
    "First GA4GH IDs:",
    ga4gh_ids[:10],
)
print(
    "Reference accessions:",
    reference_accessions[:30],
)

if (
    isinstance(mapped_variants, list)
    and mapped_variants
    and isinstance(
        mapped_variants[0],
        dict,
    )
):
    display(
        pd.json_normalize(
            mapped_variants[:5],
            max_level=2,
        )
    )

## 14. Dominios proteicos — propósito: relacionar sensibilidad funcional con regiones estructurales

Esta sección solo se ejecuta cuando exista `data/external/kcnh2_domains.csv` con columnas `domain,start,end`. Así se evita inventar coordenadas manuales sin una fuente curada.

In [ ]:
DOMAINS_PATH = (
    ROOT_DIR
    / "data"
    / "external"
    / "kcnh2_domains.csv"
)

if DOMAINS_PATH.exists():
    domains_df = pd.read_csv(
        DOMAINS_PATH
    )

    required_columns = {
        "domain",
        "start",
        "end",
    }

    missing_columns = (
        required_columns
        - set(domains_df.columns)
    )

    if missing_columns:
        raise ValueError(
            "Faltan columnas de dominios: "
            f"{sorted(missing_columns)}"
        )

    def assign_domain(
        position: int,
    ) -> str:
        matches = domains_df.loc[
            domains_df[
                "start"
            ].le(position)
            & domains_df[
                "end"
            ].ge(position),
            "domain",
        ]

        if matches.empty:
            return "Unannotated"

        return "; ".join(
            matches.astype(str)
        )

    domain_variant_df = (
        feature_df.copy()
    )

    domain_variant_df["domain"] = (
        domain_variant_df[
            "position"
        ].apply(assign_domain)
    )

    domain_summary = (
        domain_variant_df.groupby(
            "domain"
        )["score_numeric"]
        .agg(
            n="count",
            mean="mean",
            median="median",
            std="std",
        )
        .sort_values("median")
    )

    display(domain_summary)

else:
    display(
        Markdown(
            "**Análisis diferido:** aún no existe una "
            "anotación de dominios versionada y trazable."
        )
    )

## 15. Decisiones de modelamiento — propósito: convertir el EDA en una metodología justificable

El EDA debe justificar target, filtros, features y validación. La validación principal será `GroupKFold(group=position)` para evaluar generalización hacia residuos no observados.

In [ ]:
modeling_decisions = pd.DataFrame(
    {
        "decision": [
            "Target",
            "Unidad de observación",
            "Validación",
            "Baseline",
            "Representación avanzada",
            "Columnas excluidas",
            "Incertidumbre",
            "Interpretación clínica",
        ],
        "choice": [
            "score_numeric",
            "Una sustitución missense",
            "GroupKFold por position",
            (
                "Propiedades fisicoquímicas "
                "+ BLOSUM62"
            ),
            (
                "Diferencia de embeddings "
                "ESM-2 WT vs mutante"
            ),
            (
                "score, se, LLR y "
                "columnas derivadas"
            ),
            (
                "No filtrar todavía; "
                "hacer análisis de sensibilidad"
            ),
            (
                "Priorización funcional, "
                "no clasificación ACMG"
            ),
        ],
        "biological_reason": [
            (
                "Medición experimental "
                "continua"
            ),
            (
                "Atribución a un cambio "
                "único"
            ),
            (
                "Generalización a residuos "
                "no vistos"
            ),
            (
                "Cambios químicos "
                "interpretables"
            ),
            (
                "Contexto de secuencia"
            ),
            (
                "Target leakage"
            ),
            (
                "Evitar eliminar datos "
                "prematuramente"
            ),
            (
                "El tráfico no captura "
                "toda la evidencia clínica"
            ),
        ],
    }
)

display(modeling_decisions)

modeling_decisions.to_csv(
    EDA_REPORTS_DIR
    / "modeling_decisions.csv",
    index=False,
)

## 16. Exportación — propósito: guardar resultados pequeños y auditables para GitHub

Se guardan tablas resumidas y un JSON central. Los datos crudos y Parquet pesados permanecen fuera del repositorio.

In [ ]:
core_summary = {
    "reference": {
        "id": reference_record.id,
        "length": len(
            reference_sequence
        ),
    },
    "datasets": {
        "raw_rows": int(
            len(raw_scores)
        ),
        "qc_rows": int(
            len(qc_df)
        ),
        "normalized_missense_rows": int(
            len(missense_df)
        ),
    },
    "variant_classes": {
        str(row["variant_class"]): int(
            row["count"]
        )
        for _, row
        in variant_class_summary.iterrows()
    },
    "quality": {
        "numeric_scores": int(
            qc_df[
                "score_numeric"
            ].notna().sum()
        ),
        "reference_matches": int(
            qc_df[
                "ref_match"
            ].eq(True).sum()
        ),
        "reference_mismatches": int(
            qc_df[
                "ref_match"
            ].eq(False).sum()
        ),
        "protein_duplicate_rows": int(
            qc_df[
                "protein_variant_duplicate"
            ].sum()
        ),
    },
    "coverage": {
        key: (
            float(value)
            if isinstance(
                value,
                (float, np.floating),
            )
            else int(value)
        )
        for key, value
        in coverage_summary.items()
    },
    "score_medians": {
        label: float(
            class_score_summary.loc[
                label,
                "median",
            ]
        )
        for label
        in class_score_summary.index
    },
    "score_llr_spearman": float(
        spearman_llr.statistic
    ),
    "score_se_spearman": float(
        se_score_corr.statistic
    ),
}

summary_path = (
    EDA_REPORTS_DIR
    / "eda_core_summary.json"
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        core_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    f"Resumen: {summary_path}"
)
print(
    f"Tablas: {EDA_REPORTS_DIR}"
)

## 17. Conclusiones — propósito: documentar hallazgos biológicos, experimentales y metodológicos

Completa esta sección después de ejecutar el notebook y revisar sus resultados.

### Hallazgos experimentales
- ¿Qué representa el score?
- ¿Qué dirección y normalización fueron confirmadas?
- ¿Qué tan bien separa controles synonymous y stop-gained?
- ¿Qué limitaciones tiene el ensayo?

### Hallazgos bioinformáticos
- ¿Qué porcentaje del espacio missense está cubierto?
- ¿Hay posiciones sin datos?
- ¿Qué aporta `mapped_variants.json`?
- ¿Existe coherencia total con `NP_000229.1`?

### Hallazgos biológicos
- ¿Qué posiciones presentan scores bajos de forma consistente?
- ¿Qué posiciones dependen del aminoácido mutante?
- ¿Qué cambios fisicoquímicos se asocian con el score?

### Decisiones para el modelo
- Target y filtros.
- GroupKFold por posición.
- Features bioquímicas.
- Embeddings ESM-2.
- Análisis de sensibilidad por incertidumbre.

### Limitaciones
- Tráfico superficial no equivale a función electrofisiológica completa.
- Score funcional no equivale a clasificación clínica.
- HEK293 no reproduce completamente un cardiomiocito.
- La integración con ClinVar requiere armonización independiente.